# Hooks Packaging: Data Sharing and Reusable Providers

When hooks need to share data within a single invocation, or when you want to ship a bundle of hooks as one unit, Strands provides two mechanisms:

1. **`invocation_state`** - a per-request dict that any hook can read and write during one `agent(...)` call.
2. **`HookProvider`** - a class that groups related callbacks so they register and ship together.

This notebook covers both patterns and finishes with a combined example that ties together everything from the series.

> **Previous:** [01_hooks_basics.ipynb](./01_hooks_basics.ipynb) covers registration and the event lifecycle. [02_hooks_intercept.ipynb](./02_hooks_intercept.ipynb) covers writable fields for blocking, retrying, and resuming.


## Setup

This tutorial uses Claude Haiku 4.5 on Amazon Bedrock by default. It works with any Strands-supported model. Swap the `model=` argument in the `Agent(...)` constructor if you prefer a different provider (see [Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/)).


In [ ]:
%pip install -r requirements.txt --quiet

In [ ]:
import logging
import time
import uuid

from strands import Agent, tool
from strands.hooks import (
    BeforeInvocationEvent,
    AfterInvocationEvent,
    AfterModelCallEvent,
    BeforeToolCallEvent,
    AfterToolCallEvent,
    HookProvider,
    HookRegistry,
)

# Use the same Bedrock inference profile as the other 01-learn tutorials.
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# Quiet SDK logs so the hook demonstrations stay the focus of the output.
logging.getLogger("strands").setLevel(logging.WARNING)

We will reuse a simple weather tool throughout this notebook.


In [ ]:
@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city.

    Args:
        city: City name.
    """
    fake = {"Seattle": "rainy, 52F", "Austin": "sunny, 78F"}
    return fake.get(city, "clear, 70F")

## 1. Cross-hook data sharing within a single invocation

Every hook event exposes an `invocation_state` dict scoped to exactly one top-level `agent(...)` call. It is the right place to share per-request context (correlation IDs, timers, counters, open connections) across hooks.

Two complementary patterns:

1. **Stateless callbacks + `invocation_state`** - write from a `Before*` hook, read from the matching `After*` hook. Survives concurrent invocations because each call gets its own dict.
2. **`HookProvider` instance state** - store data on `self`. Convenient when callbacks need shared configuration, but requires care if the same provider is used across concurrent invocations.


### 1a. Pattern 1: share through `invocation_state`

`invocation_state` is populated from kwargs you pass to `agent(...)` plus anything hooks write to it. Any hook fired during the same invocation sees the same dict.


In [ ]:
def tag_request(event: BeforeInvocationEvent) -> None:
    event.invocation_state["request_id"] = uuid.uuid4().hex[:8]
    event.invocation_state["t0"] = time.perf_counter()
    event.invocation_state["model_calls"] = 0
    event.invocation_state["tool_calls"] = 0

def count_model(event: AfterModelCallEvent) -> None:
    event.invocation_state["model_calls"] += 1

def count_tool(event: AfterToolCallEvent) -> None:
    event.invocation_state["tool_calls"] += 1

def report(event: AfterInvocationEvent) -> None:
    s = event.invocation_state
    elapsed = time.perf_counter() - s["t0"]
    print(
        f"[{s['request_id']}] finished in {elapsed:.2f}s "
        f"({s['model_calls']} model calls, {s['tool_calls']} tool calls)"
    )

agent = Agent(model=MODEL_ID, tools=[get_weather], callback_handler=None)
for cb in (tag_request, count_model, count_tool, report):
    agent.add_hook(cb)

agent("What is the weather in Austin?")
_ = agent("And in Seattle?")  # suppress AgentResult repr in notebook output

Each line in the output shows a distinct correlation ID because `invocation_state` is freshly created per call.

You can also seed `invocation_state` from the caller:

```python
agent("Process the data", user_id="user123", session_id="sess456")
# Inside any hook: event.invocation_state["user_id"]  -> "user123"
```

This is the right channel for non-JSON-serializable objects like database connections. Those cannot live in `agent.state`, but they flow through `invocation_state` without restriction.


### 1b. Pattern 2: `HookProvider` with instance state

When you have a cohesive hook with its own configuration, package it as a `HookProvider`. The class holds configuration and (when appropriate) per-invocation counters initialized in `BeforeInvocationEvent`.


In [ ]:
class PerInvocationToolBudget(HookProvider):
    """Cap total tool calls per invocation. Resets at the start of each call."""

    def __init__(self, max_tools: int = 3):
        self.max_tools = max_tools
        self.used = 0

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self._reset)
        registry.add_callback(BeforeToolCallEvent, self._enforce)

    def _reset(self, event: BeforeInvocationEvent) -> None:
        self.used = 0

    def _enforce(self, event: BeforeToolCallEvent) -> None:
        self.used += 1
        if self.used > self.max_tools:
            event.cancel_tool = (
                f"Tool budget of {self.max_tools} exceeded for this request. "
                "Stop calling tools and answer with what you have."
            )

agent = Agent(
    model=MODEL_ID,
    tools=[get_weather],
    hooks=[PerInvocationToolBudget(max_tools=2)],
    callback_handler=None,
)
print(agent("Give me weather for Seattle, Austin, Portland, and Boise."))

The provider enforces a hard ceiling of two tool calls per `agent(...)`. The model gets a cancel message on any further tool attempt and has to answer with what it has.


## 2. Packaging multiple hooks into one `HookProvider`

A `HookProvider` groups related callbacks so they ship as a single unit. Passing one to the agent constructor (`hooks=[MyProvider()]`) calls its `register_hooks` method for you.

Below we combine everything from this series (per-request tagging, model/tool counters, a tool budget, and a single-shot summary resume) into one observability and control plugin.


In [ ]:
class LifecycleObserver(HookProvider):
    """One-stop observer: request IDs, call counts, tool budget, auto-summary."""

    def __init__(self, max_tools: int = 5, summarize: bool = False):
        self.max_tools = max_tools
        self.summarize = summarize
        # Guard must live on the instance, not on invocation_state,
        # because invocation_state is recreated on every (including resumed) invocation.
        self._summarized = False

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self._start)
        registry.add_callback(AfterModelCallEvent, self._on_model)
        registry.add_callback(BeforeToolCallEvent, self._on_tool_before)
        registry.add_callback(AfterToolCallEvent, self._on_tool_after)
        registry.add_callback(AfterInvocationEvent, self._finish)

    def _start(self, event: BeforeInvocationEvent) -> None:
        event.invocation_state.update(
            request_id=uuid.uuid4().hex[:8],
            t0=time.perf_counter(),
            model_calls=0,
            tool_calls=0,
        )

    def _on_model(self, event: AfterModelCallEvent) -> None:
        event.invocation_state["model_calls"] += 1

    def _on_tool_before(self, event: BeforeToolCallEvent) -> None:
        if event.invocation_state.get("tool_calls", 0) >= self.max_tools:
            event.cancel_tool = f"Tool budget of {self.max_tools} exceeded."

    def _on_tool_after(self, event: AfterToolCallEvent) -> None:
        event.invocation_state["tool_calls"] += 1

    async def _finish(self, event: AfterInvocationEvent) -> None:
        s = event.invocation_state
        print(
            f"[{s['request_id']}] {time.perf_counter() - s['t0']:.2f}s "
            f"model={s['model_calls']} tools={s['tool_calls']}"
        )
        if (
            self.summarize
            and not self._summarized
            and event.result
            and event.result.stop_reason == "end_turn"
        ):
            self._summarized = True
            event.resume = "Summarize your previous answer in one sentence."

agent = Agent(
    model=MODEL_ID,
    tools=[get_weather],
    hooks=[LifecycleObserver(max_tools=3, summarize=True)],
    callback_handler=None,
)
print(agent("Look up the weather in Seattle."))

One class, five hook types, reusable across agents. This is the idiomatic way to ship cross-cutting agent behavior: logging, metrics, policy, rate-limiting, etc.

For larger bundles that also want to contribute tools, system prompt fragments, or configuration, use the [`Plugin`](https://strandsagents.com/docs/user-guide/concepts/plugins/) API, which wraps `HookProvider` with additional packaging.


## Recap

Across this three-notebook series you have learned:

- The exact order of hook events in a Strands invocation, including the `After*` reverse-ordering rule ([01_hooks_basics](./01_hooks_basics.ipynb)).
- How to register a hook four ways: `agent.add_hook(fn)` (type inferred), `agent.add_hook(fn, EventType)`, `hooks=[Provider()]` in the constructor, and `agent.hooks.add_hook(Provider())` after construction ([01_hooks_basics](./01_hooks_basics.ipynb)).
- The writable fields that let hooks control the agent: `BeforeInvocationEvent.messages`, `cancel_tool`, `selected_tool`, `tool_use`, `result`, `retry` (after model / after tool), and `resume` ([02_hooks_intercept](./02_hooks_intercept.ipynb)).
- Two patterns for sharing data between hooks in one invocation: `invocation_state` (stateless callbacks) and `HookProvider` instance state.
- How to bundle multiple hooks into one reusable provider.
- That `structured_output` does not fire model-call events, and that retries may emit streaming events from discarded attempts.

### See also

- [**`13-human-in-the-loop`**](../13-human-in-the-loop/) - `BeforeToolCallEvent` is also `_Interruptible`: hooks can raise interrupts that pause the agent for human input.
- **Multi-agent hooks** - `BeforeMultiAgentInvocationEvent`, `BeforeNodeCallEvent` (with writable `cancel_node`), `AfterNodeCallEvent`, etc. Same mental model as single-agent hooks, but fired by `Graph` / `Swarm` orchestrators. See the [hooks concept guide](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/).
- [**Plugins**](https://strandsagents.com/docs/user-guide/concepts/plugins/) - package hook bundles with tools, system-prompt fragments, and config as installable units.
- [**`08-observability`**](../08-observability/) - production tracing built on the same hook surface.
